# Phase 1.1: Data Ingestion & API Exploration

**Learning Goals:**
1. Fetch data from NASA FIRMS (satellite fires), OpenAQ (AQI), and Open-Meteo (wind) APIs
2. Understand the data schemas and quality
3. Apply retry mechanisms with exponential backoff
4. Save raw data for batch processing

**North India Bounding Box:**
- West: 73.0°, South: 28.0°
- East: 78.0°, North: 33.0°

This covers Punjab, Haryana, and Delhi-NCR.

In [ ]:
# Core imports
import requests
import pandas as pd
import json
from datetime import datetime, timedelta
from tenacity import retry, stop_after_attempt, wait_exponential
import time

# North India bounding box
BBOX = {
    "west": 73.0,
    "south": 28.0,
    "east": 78.0,
    "north": 33.0
}

print("✓ Imports loaded")
print(f"Bounding Box: {BBOX}")

## 1. NASA FIRMS - Satellite Fire Detection

**API:** VIIRS SNPP NRT (Near Real-Time)

**Data includes:**
- `latitude`, `longitude` - Fire location
- `brightness` - Temperature (Kelvin)
- `frp` - Fire Radiative Power (MW)
- `acq_date`, `acq_time` - Acquisition timestamp

**Note:** You need a NASA FIRMS API key from https://firms.modaps.eosdis.nasa.gov/api/

For this learning phase, we'll use a mock key and handle the error gracefully.

In [ ]:
# NASA FIRMS fetch with retry
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def fetch_nasa_firms(bbox, map_key="DEMO_KEY", days=1):
    """
    Fetch NASA FIRMS fire data for North India.
    
    Args:
        bbox: Dict with west, south, east, north
        map_key: NASA FIRMS API key (get from https://firms.modaps.eosdis.nasa.gov/api/)
        days: Number of days to look back (1-10)
    
    Returns:
        pandas DataFrame with fire detections
    """
    bbox_str = f"{bbox['west']},{bbox['south']},{bbox['east']},{bbox['north']}"
    
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{map_key}/VIIRS_SNPP_NRT/{bbox_str}/{days}"
    
    print(f"Fetching NASA FIRMS data...")
    print(f"URL: {url[:80]}...")
    
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    
    # Parse CSV response
    from io import StringIO
    df = pd.read_csv(StringIO(response.text))
    
    # Create datetime column
    df['acq_datetime'] = pd.to_datetime(
        df['acq_date'] + ' ' + df['acq_time'].astype(str).str.zfill(4),
        format='%Y-%m-%d %H%M'
    )
    
    return df

# Try fetching (will fail with DEMO_KEY, which is expected)
try:
    firms_df = fetch_nasa_firms(BBOX, map_key="DEMO_KEY", days=1)
    print(f"\n✓ Fetched {len(firms_df)} fire detections")
    print("\nSample data:")
    print(firms_df.head())
except Exception as e:
    print(f"\n⚠️ API Error (expected with DEMO_KEY): {e}")
    print("\nCreating mock data for learning purposes...")
    
    # Create mock data
    mock_data = [
        {"latitude": 30.375, "longitude": 74.850, "brightness": 325.4, "frp": 14.2, 
         "acq_date": "2026-10-03", "acq_time": "0830"},
        {"latitude": 30.892, "longitude": 75.234, "brightness": 318.7, "frp": 12.8,
         "acq_date": "2026-10-03", "acq_time": "0845"},
        {"latitude": 31.125, "longitude": 75.671, "brightness": 331.2, "frp": 16.5,
         "acq_date": "2026-10-03", "acq_time": "0900"},
    ]
    
    firms_df = pd.DataFrame(mock_data)
    firms_df['acq_datetime'] = pd.to_datetime(
        firms_df['acq_date'] + ' ' + firms_df['acq_time'].str.zfill(4),
        format='%Y-%m-%d %H%M'
    )
    
    print("\n✓ Mock data created")
    print(firms_df)

## 2. OpenAQ - Ground Station AQI Data

**API:** OpenAQ v3 (https://api.openaq.org/v3)

**Data includes:**
- PM2.5 measurements from CPCB ground stations
- Location coordinates
- Timestamp

**Note:** OpenAQ v3 has rate limits (100 requests/5 minutes)

In [ ]:
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def fetch_openaq_locations(country_id="IN", parameter_id=2, limit=100):
    """
    Fetch OpenAQ locations in India measuring PM2.5.
    
    Args:
        country_id: Country code (IN for India)
        parameter_id: 2 for PM2.5
        limit: Max locations to fetch
    
    Returns:
        pandas DataFrame with locations
    """
    url = f"https://api.openaq.org/v3/locations"
    params = {
        "country_id": country_id,
        "parameters_id": parameter_id,
        "limit": limit
    }
    
    print(f"Fetching OpenAQ locations...")
    
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    
    data = response.json()
    locations = data.get('results', [])
    
    # Extract relevant fields
    records = []
    for loc in locations:
        if loc.get('coordinates'):
            records.append({
                'location_id': loc['id'],
                'name': loc['name'],
                'latitude': loc['coordinates']['latitude'],
                'longitude': loc['coordinates']['longitude'],
                'city': loc.get('city', 'Unknown')
            })
    
    return pd.DataFrame(records)

# Fetch locations
try:
    openaq_locations = fetch_openaq_locations(limit=50)
    
    # Filter to North India bounding box
    openaq_north_india = openaq_locations[
        (openaq_locations['latitude'] >= BBOX['south']) &
        (openaq_locations['latitude'] <= BBOX['north']) &
        (openaq_locations['longitude'] >= BBOX['west']) &
        (openaq_locations['longitude'] <= BBOX['east'])
    ]
    
    print(f"\n✓ Fetched {len(openaq_locations)} total locations")
    print(f"✓ {len(openaq_north_india)} in North India bounding box")
    print("\nNorth India locations:")
    print(openaq_north_india.head(10))
    
except Exception as e:
    print(f"\n⚠️ API Error: {e}")
    print("Creating mock data...")
    
    # Mock data
    openaq_north_india = pd.DataFrame([
        {"location_id": "IN-001", "name": "Delhi - ITO", "latitude": 28.613, "longitude": 77.209, "city": "Delhi"},
        {"location_id": "IN-002", "name": "Chandigarh Sector 25", "latitude": 30.733, "longitude": 76.779, "city": "Chandigarh"},
        {"location_id": "IN-003", "name": "Amritsar City", "latitude": 31.634, "longitude": 74.872, "city": "Amritsar"},
    ])
    print("\n✓ Mock data created")
    print(openaq_north_india)

## 3. Open-Meteo - Wind Vectors

**API:** Open-Meteo Weather Forecast API

**Data includes:**
- Wind speed (m/s)
- Wind direction (degrees)
- No API key required!

We'll fetch wind data for key cities in North India.

In [ ]:
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
def fetch_open_meteo_wind(latitude, longitude):
    """
    Fetch current wind data for a location.
    
    Args:
        latitude: Location latitude
        longitude: Location longitude
    
    Returns:
        Dict with wind speed and direction
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "wind_speed_10m,wind_direction_10m"
    }
    
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    
    data = response.json()
    current = data.get('current', {})
    
    return {
        'latitude': latitude,
        'longitude': longitude,
        'wind_speed_10m': current.get('wind_speed_10m'),
        'wind_direction_10m': current.get('wind_direction_10m'),
        'acq_datetime': current.get('time')
    }

# Key cities in North India
key_cities = [
    {"name": "Delhi", "lat": 28.6139, "lng": 77.2090},
    {"name": "Chandigarh", "lat": 30.7333, "lng": 76.7794},
    {"name": "Amritsar", "lat": 31.6340, "lng": 74.8723},
]

# Fetch wind data for each city
wind_data = []
for city in key_cities:
    try:
        wind = fetch_open_meteo_wind(city['lat'], city['lng'])
        wind['city'] = city['name']
        wind_data.append(wind)
        print(f"✓ Fetched wind data for {city['name']}")
        time.sleep(0.5)  # Rate limiting
    except Exception as e:
        print(f"⚠️ Error fetching {city['name']}: {e}")

wind_df = pd.DataFrame(wind_data)
print(f"\n✓ Fetched wind data for {len(wind_df)} cities")
print("\nWind data:")
print(wind_df)

## 4. Save Raw Data

Save the fetched data as CSV files for batch processing in the next notebook.

In [ ]:
import os

raw_data_dir = "/home/jovyan/work/data/raw"
os.makedirs(raw_data_dir, exist_ok=True)

# Save each dataset
firms_df.to_csv(f"{raw_data_dir}/nasa_firms_fires.csv", index=False)
print(f"✓ Saved {len(firms_df)} fire records to nasa_firms_fires.csv")

openaq_north_india.to_csv(f"{raw_data_dir}/openaq_locations.csv", index=False)
print(f"✓ Saved {len(openaq_north_india)} AQI locations to openaq_locations.csv")

wind_df.to_csv(f"{raw_data_dir}/open_meteo_wind.csv", index=False)
print(f"✓ Saved {len(wind_df)} wind records to open_meteo_wind.csv")

print(f"\n✅ All raw data saved to {raw_data_dir}")

## Summary

**What we learned:**
1. ✅ Fetched data from 3 external APIs
2. ✅ Applied retry mechanisms with exponential backoff (`tenacity`)
3. ✅ Understood data schemas and quality
4. ✅ Saved raw data for batch processing

**Data collected:**
- 🔥 NASA FIRMS: Satellite fire detections (lat, lng, brightness, FRP)
- 🌫️ OpenAQ: Ground station locations (lat, lng, PM2.5 capability)
- 💨 Open-Meteo: Wind vectors (speed, direction)

**Next notebook:** `01_spatial_indexing_h3.ipynb`
- Learn H3 spatial indexing at Resolution 7
- Map coordinates to hexagonal cells
- Aggregate data by H3 index